In [ ]:
import os
import torch
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset ,DataLoader,Subset
from PIL import Image
import kagglehub
import torch.nn as nn


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'plantvillage' dataset.
Path to dataset files: /kaggle/input/plantvillage


In [ ]:
torch.manual_seed(42)
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device {device}")

using device cuda


#**Random Seed Working process:**  
A random seed is a fixed value used to control randomness so that the same random results can be reproduced when the code is run again.


Google Colab is a platform that allows you to use Google's cloud computers over the internet.

In [ ]:
Train_path= os.path.join(path,"PlantVillage","train")
Val_path= os.path.join(path,"PlantVillage","val")

In [ ]:
transfrom = transforms.Compose(
    [
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5]*3 , std=[0.5]*3) # RGB hence = *3
    ]
)

# Custom Dataset

In [ ]:
class Mulitclassclassification(Dataset):
  def __init__(self,root_dir,transform=None):
    super().__init__()
    self.samples  = []
    self.transform = transform
    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])

# root directory holo train or teke nicce sob file like Apple___healthy

    self.class_to_idx = {cls_name : idx for idx ,cls_name in enumerate(self.classes)}

    for class_name in self.classes:
      class_path = os.path.join(root_dir,class_name)

      for img_name in os.listdir(class_path):           # root directory "train/Apple___healthy" or teke file/image ber korbe akeake
        img_path = os.path.join(class_path,img_name)     # img_path = train/Apple___healthy/img1.jpg

        if os.path.isfile(img_path):

          label =self.class_to_idx[class_name]               #  label = self.class_to_idx["Apple___rust"]  label = 1
          self.samples.append((img_path,label))



  def __len__(self):

    return len(self.samples)

  def __getitem__(self,index):


    img_path,label = self.samples[index]
    image = Image.open(img_path).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image,label



In [ ]:
train_dataset_full= Mulitclassclassification(Train_path,transfrom)
val_dataset_full= Mulitclassclassification(Val_path,transfrom)
num_classes= len(train_dataset_full.classes)
print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset_full))
print("Full val size:", len(val_dataset_full))


Number of classes: 38
Full train size: 43444
Full val size: 10861


In [ ]:
pin = True if device.type == 'cuda' else False

train_loader = DataLoader(
    train_dataset_full,
    batch_size=32,
    shuffle=True,
    pin_memory=pin
)

val_loader = DataLoader(
    val_dataset_full,
    batch_size=32,
    shuffle=False,
    pin_memory=pin
)

In [ ]:
class MyCNN(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(2),
        nn.Conv2d(32,64,kernel_size=3,padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(2),
        nn.Conv2d(64,128,kernel_size=3,padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(2)

    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*16*16,128),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(64,num_classes)

    )
  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x


In [ ]:
model = MyCNN(num_classes=num_classes).to=device)

In [ ]:
learning_rate = 0.001
epochs =10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=learning_rate)



In [ ]:
for epoch in range(epochs):
  model.train()
  total_loss=0
  for batch_features,labels in train_loader:
    batch_features = batch_features.to(device)
    labels=labels.to(device)
    outputs = model(batch_features)
    loss = criterion(outputs,labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  avg_loss = total_loss / len(train_loader)
  print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")



Epoch 1/10, Loss: 1.9858
Epoch 2/10, Loss: 1.2867
Epoch 3/10, Loss: 1.0259
Epoch 4/10, Loss: 0.8529
Epoch 5/10, Loss: 0.7097
Epoch 6/10, Loss: 0.5999
Epoch 7/10, Loss: 0.5392
Epoch 8/10, Loss: 0.4956
Epoch 9/10, Loss: 0.4410
Epoch 10/10, Loss: 0.4094


In [ ]:
def evaluate(loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for batch_features, batch_labels in loader:

            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            output = model(batch_features)

            pred = output.argmax(1)

            correct += (pred == batch_labels).sum().item()
            total += batch_labels.size(0)

    return correct / total




In [ ]:
val_acc = evaluate(val_loader)

print("Test Accuracy:", val_acc)

Test Accuracy: 0.9442961053310008
